<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/03a_visualizations_abigail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install contextily

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np

#added more that we use in lab
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!ls /content/drive
!ls /content/drive/Shareddrives

MyDrive  Shareddrives


In [5]:
import pandas as pd

gdf = pd.read_csv("/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/cleaned/rbl_total_cleaned.csv")

In [6]:
gdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148940 entries, 0 to 148939
Data columns (total 40 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         148940 non-null  int64  
 1   uniqueid                           148940 non-null  object 
 2   business_account_number            148940 non-null  int64  
 3   location_id                        148940 non-null  object 
 4   ownership_name                     148940 non-null  object 
 5   dba_name                           148787 non-null  object 
 6   street_address                     148940 non-null  object 
 7   city                               148940 non-null  object 
 8   state                              148934 non-null  object 
 9   source_zipcode                     148931 non-null  float64
 10  business_start_date                148940 non-null  object 
 11  business_end_date                  7551

In [17]:
# # adding another plot and importing a shpfile of SF so we can see where the points are in SF

# import matplotlib.pyplot as plt

# # Importing SF geometry
# # URL for 2025 TIGER/Line Place boundaries - info here on
# ## how to use: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]


# eproject to same as our gdf
sf_poly = sf_poly.to_crs(epsg=4326)

In [7]:
import plotly.express as px

# counting number of opening and closings - Abigail
counts = gdf.groupby(["year", "status"]).size().reset_index(name="count")
counts = counts[counts["year"] <= 2025]

# Making graph of this
fig = px.line(
    counts,
    x="year",
    y="count",
    color="status",
    markers=True
)

# Adding labels
fig.update_layout(
    title="San Francisco Business Openings vs Closings (2016–2025)",
    xaxis_title="Year",
    yaxis_title="Number of Businesses",
    legend_title="Status"
)

#adding code to show each year on x-axis
fig.update_xaxes(dtick=1)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

#making background white and removing vert lines
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray"
)

fig.show()


## **Tract Level Analysis**

In [8]:
#I added a shp file with census tracts to our folder

tracts_gdf = gpd.read_file(
    "/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/cb_2020_06_tract_500k"
)

In [9]:
#checking to make sure all good

tracts_gdf.columns

# simplifying
tracts_gdf = tracts_gdf[["NAME", "NAMELSAD", "STATE_NAME","GEOID", "geometry"]]


In [10]:
# was getting error with "index_right" - so troubleshooted this
tracts_gdf = tracts_gdf.drop(columns=["index_right", ], errors="ignore")
gdf = gdf.drop(columns=["index_right"], errors="ignore")

In [11]:
# Joining the gdf and tract gdf so we can summarize within tracts

gdf = gdf.set_geometry(
    gpd.points_from_xy(gdf.lon, gdf.lat)
)

gdf = gdf.set_crs(epsg=4326)
tracts_gdf = tracts_gdf.to_crs(epsg=4326)

# joining within
business_tracts_gdf = gpd.sjoin(gdf, tracts_gdf, how="left", predicate="within")

In [12]:
business_tracts_gdf = business_tracts_gdf[(business_tracts_gdf["year"] >= 2016) & (business_tracts_gdf["year"] <= 2025)]

In [13]:
business_tracts_gdf.columns.tolist()

cols_to_keep = [
    "uniqueid",
    "naics_code",
    "year",
    "status",
    "location_start_date",
    "location_end_date",
    "GEOID",
    "geometry",
    "lon", "lat"
]

business_tracts_gdf = business_tracts_gdf[cols_to_keep]

In [14]:
# I'm counting the number of businesses per tract per year, separated by closing and openings status
tract_year = (
    business_tracts_gdf
    .groupby(["GEOID", "year", "status"])
    .size()
    .reset_index(name="count")
    .pivot(index=["GEOID", "year"], columns="status", values="count")
    .fillna(0)
    .reset_index()
)

In [15]:
tracts_plot = tracts_gdf[["GEOID", "geometry"]].merge(
    tract_year,
    on="GEOID",
    how="left"
).fillna(0)

In [16]:
tracts_plot.info()
tracts_plot = gpd.clip(tracts_plot, sf_poly)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 11301 entries, 0 to 11300
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   GEOID     11301 non-null  object  
 1   geometry  11301 non-null  geometry
 2   year      11301 non-null  float64 
 3   closed    11301 non-null  float64 
 4   opened    11301 non-null  float64 
dtypes: float64(3), geometry(1), object(1)
memory usage: 441.6+ KB


NameError: name 'sf_poly' is not defined

In [ ]:
tracts_plot["year"] = tracts_plot["year"].astype(int)
tracts_plot = tracts_plot.sort_values(["GEOID", "year"])

In [ ]:
opened = tracts_plot[tracts_plot["status"] == "opened"]
closed = tracts_plot[tracts_plot["status"] == "closed"]

In [ ]:
# fig_closed = px.choropleth_mapbox(
#     closed,
#     geojson=closed.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="count",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},  # centering on SF becuase it wasn't when I tested
#     color_continuous_scale="Reds",
#     height=600
# )

# fig_closed.update_layout(title="Business Closings by Census Tract (2016–2025)")
# fig_closed.show()

In [ ]:
# fig_opened = px.choropleth_mapbox(
#     opened,
#     geojson=closed.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="count",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},  # centering on SF becuase it wasn't when I tested
#     color_continuous_scale="Blues",
#     height=600
# )

# fig_opened.update_layout(title="Business Openings by Census Tract (2016–2025)")
# fig_opened.show()

Need to baseline so we don't have viz skewed by areas with higher # of overall businesses

In [ ]:
# Reformatting so each row is a unique count per tract, year, and status (so we don't have closings and openings for busness in one row)
business_tracts_separated = (
    business_tracts_gdf
    .groupby(["GEOID", "year", "status"])
    .size()
    .reset_index(name="count")
)

In [ ]:
# Baselining by 2016 (but this can change - just testing it out)

baseline = business_tracts_separated[business_tracts_separated["year"] == 2016][["GEOID", "status", "count"]].rename(columns={"count": "baseline"}) # makes new df and filtering for only 2016 so we can use as baseline
business_tracts_separated = business_tracts_separated.merge(baseline, on=["GEOID", "status"], how="left") # joining this new baseline df to business tracts df so we can calculate change
business_tracts_separated["change_from_2016"] = business_tracts_separated["count"] - business_tracts_separated["baseline"] # subtracts change from baseline


In [ ]:
tracts_plot = tracts_gdf[["GEOID", "geometry"]].merge(
    business_tracts_separated,
    on="GEOID",
    how="left"
).fillna(0)

In [ ]:
opened = tracts_plot[tracts_plot["status"] == "opened"]
closed = tracts_plot[tracts_plot["status"] == "closed"]

In [ ]:
# sanity check on changes

opened["change_from_2016"].describe()

# most tracts didn't have major changes (average of 3 fewer openings compared to 2016)
# - but some tracts had a major change (both pos and neg compared to 2016)- these extremes will skew our viz

In [ ]:
closed["change_from_2016"].describe()

In [ ]:
# fig = px.choropleth_mapbox(
#     opened,
#     geojson=tracts_plot.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="change_from_2016",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=11,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="Blues",
#     range_color=[-20, 20], # adding this so that the legend scale doesn't change each year - need to make decision around this bc min and max will not show most changes
#     height=600,
# )

# fig.update_layout(title="SF Business Openings Change from 2016 by Census Tract")
# fig.show()

In [ ]:
fig.write_html("sf_business_map.html")

In [ ]:
from google.colab import files
files.download("sf_business_map.html")

In [ ]:
# fig = px.choropleth_mapbox(
#     closed,
#     geojson=tracts_plot.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="change_from_2016",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=11,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="Reds",
#     range_color=[-10,20],
#     height=600,
# )

# fig.update_layout(title="SF Business Closings Change from 2016 by Census Tract")
# fig.show()

In [ ]:
# openings = business_tracts_gdf[business_tracts_gdf["status"] == "opened"]
# closings = business_tracts_gdf[business_tracts_gdf["status"] == "closed"]

In [ ]:
# # counting by tract and by year for openings and closings

# openings_business = (
#     openings
#     .groupby(["GEOID_right", "year"])
#     .size()
#     .reset_index(name="opened")
# )

# closings_business = (
#     closings
#     .groupby(["GEOID_right", "year"])
#     .size()
#     .reset_index(name="closed")
# )

In [ ]:
# # joining them back together

# tract_year = openings_business.merge(
#     closings_business,
#     on=["GEOID_right", "year"],
#     how="outer"
# ).fillna(0)

In [ ]:
# tract_year = tract_year.rename(columns={"GEOID_right": "GEOID"})
# tract_year

In [ ]:
# map_2016 = tracts_gdf.merge(
#     tract_year[tract_year["year"] == 2016],
#     on="GEOID",
#     how="left"
# )

# map_2016.plot(column="closed", legend=True)